# TechStream - Análisis Exploratorio de Datos (EDA)

Este notebook realiza un análisis exploratorio del dataset de telemetría de servidores generado sintéticamente. El objetivo es comprobar la coherencia física de las variables de los sensores y entender la distribución y correlación de los datos antes de diseñar el modelo de Deep Learning en PyTorch.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar matplotlib para que funcione en entornos headless si es necesario
import matplotlib
matplotlib.use('Agg')
%matplotlib inline

logger_format = "%(asctime)s - %(levelname)s - %(message)s"
import logging
logging.basicConfig(level=logging.INFO, format=logger_format)
logger = logging.getLogger(__name__)

logger.info("Librerías importadas correctamente.")

## 1. Carga y Estructura de Datos

In [ ]:
df_path = "dataset_servidores.csv"
if os.path.exists(df_path):
    df = pd.read_csv(df_path)
    logger.info(f"Dataset cargado con {df.shape[0]} filas y {df.shape[1]} columnas.")
else:
    logger.error(f"No se encontró el dataset en {df_path}. Por favor ejecuta el generador primero.")

df.head()

## 2. Estadísticas Descriptivas y Análisis de Nulos

In [ ]:
# Información general de tipos y nulos
df.info()

# Estadísticas de variables continuas
df.describe()

## 3. Distribución del Target (Desbalanceo de Clases)

Dado que buscamos anomalías en servidores, es normal esperar que el dataset esté desbalanceado hacia la clase mayoritaria (funcionamiento normal `Fallo = 0`). Analicemos la proporción.

In [ ]:
class_counts = df['failure'].value_counts()
class_percentages = df['failure'].value_counts(normalize=True) * 100

print("Conteo de clases:")
print(class_counts)
print("\nPorcentaje de clases:")
print(class_percentages)

plt.figure(figsize=(6, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette='viridis')
plt.title('Distribución de Clases (Fallo vs Normal)')
plt.xlabel('Fallo (0 = Normal, 1 = Fallo)')
plt.ylabel('Cantidad de Registros')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig('distribution_clases.png', bbox_inches='tight')
plt.show()

## 4. Análisis de Correlación

Para verificar la coherencia de las leyes físicas que simulamos, observemos la correlación lineal de Pearson entre variables continuas y el target.

In [ ]:
numeric_cols = ['cpu_usage', 'mem_usage', 'network_traffic', 'cpu_temp', 'failure']
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlación de Pearson')
plt.savefig('matriz_correlacion.png', bbox_inches='tight')
plt.show()

## 5. Distribución de Variables Clave en función de la presencia de Fallos

Visualicemos cómo se comportan la temperatura y el uso de memoria cuando hay fallos del sistema vs funcionamiento normal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de Temperatura de la CPU según Fallo
sns.histplot(data=df, x='cpu_temp', hue='failure', kde=True, ax=axes[0], palette='Set1', bins=40, alpha=0.6)
axes[0].set_title('Distribución de Temperatura de CPU por Estado')
axes[0].set_xlabel('Temperatura CPU (°C)')
axes[0].set_ylabel('Frecuencia')

# Boxplot de Uso de Memoria según Fallo
sns.boxplot(data=df, x='failure', y='mem_usage', ax=axes[1], palette='Set2')
axes[1].set_title('Uso de Memoria por Estado')
axes[1].set_xlabel('Estado (0 = Normal, 1 = Fallo)')
axes[1].set_ylabel('Uso Memoria (%)')

plt.savefig('distribucion_variables_fallo.png', bbox_inches='tight')
plt.show()

## 6. Conclusiones e Implicaciones para el Modelo

1. **Desbalanceo de Clases:** Hemos verificado un claro desbalanceo (pocos fallos comparado con datos normales). Esto implica que al entrenar la red neuronal en PyTorch debemos usar técnicas de ponderación de pérdida como `pos_weight` en `BCEWithLogitsLoss` o un `WeightedRandomSampler` en el loader.
2. **Dependencia Temporal:** Dado que la inercia térmica hace que el calor se acumule con el tiempo y que la lógica del fallo requiere sobrecalentamiento durante varios pasos consecutivos, una red que analice una única muestra individual podría perder contexto temporal. Por tanto, utilizaremos una **ventana deslizante** (ej: agrupar las últimas 5 muestras del servidor) para proveer contexto secuencial al MLP.